# Step 5 — Neural Ranker: History Attention, Slate Context, Ordinal Output

**Purpose.** Steps 2–4d showed that scalar feature engineering on a tree model has plateaued at NDCG@10 ≈ 0.416, and pointed to two reasons:

1. **Information the tree can't use in scalar form.** The user's semantic history profile scores NDCG@10 0.116 on its own, yet as a single number it is not significant in the model (Step 4d).
2. **Representation, not information.** `slate_pct_pop`, a within-slate rank with no new information, took 48% of the model's split gain (Step 4b) and is now its most important feature (Step 4d).

This model addresses both directly, and finally delivers the output the interim report promised (§1.2.2): cumulative probabilities P(Y ≥ 1), P(Y ≥ 2), P(Y ≥ 3).

| Component | What it does | Motivated by |
|---|---|---|
| Feature encoder | MLP over the engineered features (32-feature set from Step 4d, or all 55) | Steps 2–4d |
| Item and query embeddings | The 64-d LSA vectors from Step 4, kept fixed | Step 4 |
| **History attention** (DIN, Zhou et al. 2018) | For each candidate, weights the user's last 50 history events by relevance to *that* candidate, using each event's item embedding, interaction type and age | Step 4: semantic history matters, but not as one averaged number |
| **Slate context** | Each candidate sees the mean representation of its own slate and its difference from it | Step 4b/4d: within-slate comparison is what the tree model leaned on most |
| **Ordinal head** | One score *s* per candidate and three learned, ordered thresholds: P(Y ≥ k) = σ(s − θₖ). The probabilities are consistent by construction, and ranking by *s* equals ranking by expected interaction level | Interim report §1.2.2 |

**Training.** Loss = ordinal binary cross-entropy (one term per threshold) + a listwise softmax loss over each slate with gains 2^label − 1, which matches how NDCG@10 is scored. No position input, and the architecture is permutation-invariant within a slate, so the list-order leakage from the interim report cannot enter.

**Ablation (each run adds one component):** MLP on features → + embeddings and history attention → + slate context → full model on all 55 features. Every run is compared with the LightGBM benchmark (NDCG@10 0.4159).

**Runtime:** Colab **L4 GPU** (24 GB GPU memory and enough system RAM). First run builds and caches the embeddings (~15 min); then roughly 10–20 min per training run.

In [1]:
import os, time, gc, copy
from pathlib import Path
import numpy as np
import pandas as pd
import torch, torch.nn as nn, torch.nn.functional as Fn
import pyarrow as pa, pyarrow.compute as pc, pyarrow.csv as pacsv, pyarrow.parquet as pq, pyarrow.dataset as ds
import scipy.sparse as sp
from scipy.sparse.linalg import svds

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = Path('/content/drive/MyDrive/JDSearch/Raw Data')
except ImportError:
    DATA_DIR = Path(os.environ.get('JD_DATA_DIR', '~/Downloads/JDsearch')).expanduser()
OUT_DIR = DATA_DIR / 'processed'

SMOKE = os.environ.get('JD_SMOKE') == '1'      # local smoke test: ~2% of users, one epoch, CPU
SEED, K, L = 42, 64, 50                         # embedding size, history length per user
EPOCHS, BATCH = (1, 32) if SMOKE else (4, 48)   # BATCH = slates per step
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
USE_AMP = DEVICE == 'cuda'
torch.manual_seed(SEED); np.random.seed(SEED)
print(f'device {DEVICE}' + (f' ({torch.cuda.get_device_name(0)})' if USE_AMP else '') + (' | SMOKE TEST' if SMOKE else ''))
if DEVICE == 'cpu' and not SMOKE:
    print('WARNING: no GPU. Runtime > Change runtime type > L4 GPU, or training will take hours.')

FEATURES_LEAN = [   # Step 4d: 25 significant features (near-duplicates removed) + 7 deduplicated user-level features
    'slate_pct_pop', 'i_users_click_7d', 'uc_days_since_any', 'i_users_cart_90d', 'i_users_ord_365d',
    'q_name_tfidf_sim', 'slate_len', 'ubrand_days_since_any', 'c3_click_7d', 'slate_cat3_share',
    'u_days_since_click', 'qc_cat3_prior_max', 'uc_click_7d', 'ucat3_days_since_any', 'q_name_match_frac',
    'slate_pct_text', 'slate_pct_sem', 'c3_ord_per_cart_90d', 'q_meta_tfidf_sim', 'q_sem_lsa_sim',
    'u_days_since_cart', 'q_n_terms', 'uc_cart_90d', 'ucat2_click_7d', 'uc_ord_365d',
    'u_ord_365d', 'u_click_7d', 'u_days_since_ord', 'u_cart_per_click_7d', 'u_ord_per_cart_90d',
    'u_queryless_share_7d', 'u_query_repeat_rate']
SEASONAL = ['cat3_festival_lift', 'u_festival_share', 'ucat3_fest_ord', 'ucat3_fest_share']
FEATURES_ALL = [c for c in pq.read_schema(OUT_DIR / 'features.parquet').names
                if c not in ('user', 'wid', 'label', 'split') and c not in SEASONAL]
missing = [f for f in FEATURES_LEAN if f not in FEATURES_ALL]
if missing:
    print(f'WARNING: {missing} not in features.parquet (run Steps 4 and 4b first); continuing without them')
    FEATURES_LEAN = [f for f in FEATURES_LEAN if f in FEATURES_ALL]
print(f'feature sets: lean {len(FEATURES_LEAN)} | all {len(FEATURES_ALL)}')

LGBM_REF = {'NDCG@10': 0.4159, 'HR@10': 0.7196, 'Purchase-HR@10': 0.6692, 'cold NDCG@10': 0.1138}   # Step 4b test set

Mounted at /content/drive
device cuda (NVIDIA L4)
feature sets: lean 32 | all 55


## 1. Load the feature table

Rows are sorted by user, so each slate is a contiguous block. Only slates with at least one interaction are kept (as in every earlier benchmark), giving the same 25,822 test slates.

In [2]:
nU = len(pd.read_parquet(OUT_DIR / 'users.parquet', columns=['user']))
in_scope = np.random.default_rng(SEED + 2).random(nU) < 0.02 if SMOKE else np.ones(nU, bool)
scope_users = np.flatnonzero(in_scope)

dset = ds.dataset(OUT_DIR / 'features.parquet')
flt = ds.field('user').isin(pa.array(scope_users.astype(np.int32))) if SMOKE else None
feat = dset.to_table(columns=['user', 'wid', 'label', 'split'] + FEATURES_ALL, filter=flt).to_pandas()
feat = feat.sort_values('user', kind='stable').reset_index(drop=True)
feat = feat[feat.groupby('user')['label'].transform('max') > 0].reset_index(drop=True)
USER = feat['user'].to_numpy().astype(np.int64)
LABEL = feat['label'].to_numpy().astype(np.int64)
SPLIT = feat['split'].to_numpy()
print(f'{len(feat):,} candidate rows | slates: ' +
      ', '.join(f'{s} {feat.loc[SPLIT == s, "user"].nunique():,}' for s in ['train', 'val', 'test']))

15,348,717 candidate rows | slates: train 120,081, val 25,825, test 25,822


## 2. LSA item and query embeddings (built once, cached on Drive)

Step 4 computed these in memory but didn't save them. This cell rebuilds them with the identical procedure and seed (TF-IDF over product names, truncated SVD with k = 64) and caches them in `processed/`, so later runs load them in seconds. Rows are L2-normalised.

In [3]:
sfx = '_smoke' if SMOKE else ''
CACHE = {k: OUT_DIR / f'lsa{K}_{k}{sfx}.npy' for k in ('wids', 'item', 'query')}

def safe_save(path, arr):
    tmp = path.with_suffix('.tmp')
    with open(tmp, 'wb') as f:
        np.save(f, arr)
    np.load(tmp, mmap_mode='r')                       # verify it reads back before replacing
    os.replace(tmp, path)

def l2rows(M):
    n = np.linalg.norm(M, axis=1, keepdims=True); n[n == 0] = 1
    return M / n

def build_lsa():
    cw = pd.read_parquet(OUT_DIR / 'candidates.parquet', columns=['user', 'wid'])
    ew = pd.read_parquet(OUT_DIR / 'events.parquet', columns=['user', 'wid', 'type'])
    ew = ew[ew['type'] < 3]
    need = np.union1d(cw['wid'].to_numpy()[in_scope[cw['user'].to_numpy()]],
                      ew['wid'].to_numpy()[in_scope[ew['user'].to_numpy()]])
    del cw, ew; gc.collect()

    wid_parts, doc_parts, tok_parts, off = [], [], [], 0
    reader = pacsv.open_csv(DATA_DIR / 'product_meta_data.txt',
        parse_options=pacsv.ParseOptions(delimiter='\t', quote_char=False),
        convert_options=pacsv.ConvertOptions(include_columns=['wid', 'name'],
                                             column_types={'wid': pa.int64(), 'name': pa.string()}))
    for batch in reader:
        tb = pa.Table.from_batches([batch])
        wid = tb['wid'].to_numpy()
        j = np.searchsorted(need, wid)
        keep = (j < len(need)) & (need[np.minimum(j, len(need) - 1)] == wid)
        if not keep.any():
            continue
        tb = tb.filter(pa.array(keep))
        parts = pc.split_pattern(pc.fill_null(tb['name'], '-1'), '\x18')
        flat = pc.list_flatten(parts)
        flat = pc.if_else(pc.equal(flat, ''), '-1', flat)
        ids = np.arange(off, off + len(tb), dtype=np.int64); off += len(tb)
        wid_parts.append(tb['wid'].to_numpy())
        doc_parts.append(np.repeat(ids, pc.list_value_length(parts).to_numpy()))
        tok_parts.append(pc.cast(flat, pa.int64()).to_numpy())
    wids = np.concatenate(wid_parts)
    d_, t_ = np.concatenate(doc_parts), np.concatenate(tok_parts)
    m = t_ != -1; d_, t_ = d_[m], t_[m]
    nD = len(wids)

    vocab = np.unique(t_)
    key = np.unique(d_ * len(vocab) + np.searchsorted(vocab, t_))
    d_, c_ = key // len(vocab), key % len(vocab)
    idf = (np.log((nD + 1) / (np.bincount(c_, minlength=len(vocab)) + 1)) + 1).astype(np.float32)
    norm = np.sqrt(np.bincount(d_, weights=idf[c_] ** 2, minlength=nD)); norm[norm == 0] = 1
    A = sp.csr_matrix((idf[c_] / norm[d_], (d_, c_)), shape=(nD, len(vocab)), dtype=np.float32)
    del d_, c_, key, t_; gc.collect()
    v0 = np.random.default_rng(SEED).random(min(A.shape)).astype(np.float32)
    _, S, Vt = svds(A, k=K, v0=v0, return_singular_vectors='vh')
    Vt = Vt[np.argsort(-S)].astype(np.float32)
    item = np.empty((nD, K), np.float32)
    for s in range(0, nD, 1_000_000):
        item[s:s + 1_000_000] = A[s:s + 1_000_000] @ Vt.T
    item = l2rows(item).astype(np.float16)
    del A; gc.collect()

    users = pd.read_parquet(OUT_DIR / 'users.parquet', columns=['user', 'qry'])
    queries = pd.read_parquet(OUT_DIR / 'queries.parquet')
    qs = users['qry'].map(queries.set_index('qry')['text']).str.split('\x18')
    q_doc = np.repeat(np.arange(len(users), dtype=np.int64), qs.str.len().to_numpy())
    q_tok = pd.to_numeric(qs.explode(), errors='coerce').fillna(-1).to_numpy(np.int64)
    pos = np.searchsorted(vocab, q_tok)
    ok = (q_tok != -1) & (pos < len(vocab)) & (vocab[np.minimum(pos, len(vocab) - 1)] == q_tok)
    kq = np.unique(q_doc[ok] * len(vocab) + pos[ok])
    Q = sp.csr_matrix((idf[kq % len(vocab)], (kq // len(vocab), kq % len(vocab))),
                      shape=(len(users), len(vocab)), dtype=np.float32)
    qemb = l2rows(Q @ Vt.T).astype(np.float16)
    return wids, item, qemb

t0 = time.time()
if all(p.exists() for p in CACHE.values()):
    emb_wids, item_emb, q_emb = (np.load(CACHE[k]) for k in ('wids', 'item', 'query'))
    print(f'loaded cached embeddings in {time.time() - t0:.0f}s')
else:
    emb_wids, item_emb, q_emb = build_lsa()
    for k, a in zip(('wids', 'item', 'query'), (emb_wids, item_emb, q_emb)):
        safe_save(CACHE[k], a)
    print(f'built and cached embeddings in {time.time() - t0:.0f}s')
print(f'item embeddings {item_emb.shape} | query embeddings {q_emb.shape}')
emb_index = pd.Index(emb_wids)
DOC = emb_index.get_indexer(feat['wid']) + 1            # 0 = no name / unknown (zero vector)

built and cached embeddings in 679s
item embeddings (11587364, 64) | query embeddings (173831, 64)


## 3. User history sequences

The last 50 history events of each user (clicks, carts and orders; follows are <0.1%), most recent first: item, interaction type, and age in days (log-scaled). Age matters because each type is only recorded within its retention window (clicks 7 d, carts 90 d, orders 365 d; Step 1), and the attention can learn to read type and age together.

In [4]:
ev = pd.read_parquet(OUT_DIR / 'events.parquet', columns=['user', 'pos', 'wid', 'type', 'age_days'])
ev = ev[(ev['type'] < 3) & in_scope[ev['user'].to_numpy()]]
ev = ev.sort_values(['user', 'pos'], kind='stable')
slot = ev.groupby('user').cumcount(ascending=False).to_numpy()      # 0 = most recent
ev, slot = ev[slot < L], slot[slot < L]
u_ev = ev['user'].to_numpy()
H_DOC = np.zeros((nU, L), np.int64); H_TYPE = np.zeros((nU, L), np.int64)
H_AGE = np.zeros((nU, L), np.float32); H_MASK = np.zeros((nU, L), bool)
H_DOC[u_ev, slot] = emb_index.get_indexer(ev['wid']) + 1
H_TYPE[u_ev, slot] = ev['type'].to_numpy()
H_AGE[u_ev, slot] = np.log1p(ev['age_days'].to_numpy())
H_MASK[u_ev, slot] = True
del ev, slot, u_ev; gc.collect()

# keep only the embedding rows actually used, so the table fits comfortably on the GPU
used = np.unique(np.r_[0, DOC, H_DOC[H_MASK]])
E_full = np.vstack([np.zeros((1, K), np.float16), item_emb])
E = torch.from_numpy(E_full[used]).to(DEVICE)
DOC = np.searchsorted(used, DOC); H_DOC = np.searchsorted(used, H_DOC)
QE = torch.from_numpy(q_emb).to(DEVICE)
del E_full, item_emb; gc.collect()
print(f'history: users with events {H_MASK.any(1)[scope_users].mean():.1%} | mean length {H_MASK[scope_users].sum(1).mean():.1f} '
      f'| embedding table on {DEVICE}: {E.shape[0]:,} x {K}')

history: users with events 100.0% | mean length 39.5 | embedding table on cuda: 9,209,446 x 64


## 4. Feature preprocessing

Counts and recencies are heavy-tailed, so each feature gets a signed log transform, then is standardised with **training-split** statistics. Missing values become 0 plus a missing-indicator column, so "never interacted" stays distinguishable from "interacted long ago".

In [5]:
def build_X(cols):
    X = feat[cols].to_numpy(np.float32, copy=True)
    nan = np.isnan(X)
    np.copysign(np.log1p(np.abs(X)), X, out=X)
    tr = SPLIT == 'train'
    mu = np.nanmean(X[tr], 0); sd = np.nanstd(X[tr], 0); sd[~(sd > 1e-6)] = 1; mu = np.nan_to_num(mu)
    X -= mu; X /= sd; X[nan] = 0
    has_nan = nan.any(0)
    X = np.concatenate([X, nan[:, has_nan].astype(np.float32)], 1)
    return torch.from_numpy(X)

def make_split(name):
    idx = np.flatnonzero(SPLIT == name)
    u = USER[idx]
    start = np.r_[0, np.flatnonzero(u[1:] != u[:-1]) + 1]
    return {'idx': idx, 'start': start, 'size': np.diff(np.r_[start, len(idx)]), 'user': u[start]}

SPLITS = {s: make_split(s) for s in ['train', 'val', 'test']}
X_CACHE = {}
def get_X(key):
    if key not in X_CACHE:
        X_CACHE.clear(); gc.collect()                 # keep one feature matrix in memory at a time
        X_CACHE[key] = build_X(FEATURES_LEAN if key == 'lean' else FEATURES_ALL)
    return X_CACHE[key]
LABEL_T, DOC_T = torch.from_numpy(LABEL), torch.from_numpy(DOC)
H_DOC_T, H_TYPE_T = torch.from_numpy(H_DOC), torch.from_numpy(H_TYPE)
H_AGE_T, H_MASK_T = torch.from_numpy(H_AGE), torch.from_numpy(H_MASK)

## 5. Model

Shapes: B slates per batch, M candidates per slate (padded), L = 50 history events, d = 128.

The DIN attention score for history event *h* and candidate *c* is an MLP over [h, c, h − c, h ⊙ c]. Its first layer is computed in the algebraically identical split form W₁h + W₂c + W₃(h ⊙ c), which avoids materialising the B × M × L × 256 concatenation.

In [6]:
class Ranker(nn.Module):
    def __init__(self, n_feat, cfg, d=128, k=K, h=64):
        super().__init__()
        self.cfg = cfg
        self.feat = nn.Sequential(nn.Linear(n_feat, d), nn.ReLU(), nn.Linear(d, d), nn.ReLU())
        width = d
        if cfg['emb']:
            self.item, self.qry = nn.Linear(k, h), nn.Linear(k, h)
            width += 2 * h
        if cfg['history']:
            self.type_emb, self.age = nn.Embedding(3, 8), nn.Linear(1, 8)
            self.hist = nn.Linear(h + 16, h)
            self.att_h, self.att_c, self.att_p = nn.Linear(h, h), nn.Linear(h, h, bias=False), nn.Linear(h, h, bias=False)
            self.att_out = nn.Linear(h, 1)
            width += h
        self.enc = nn.Sequential(nn.Linear(width, d), nn.ReLU())
        self.head = nn.Sequential(nn.Linear(3 * d if cfg['slate_ctx'] else d, d), nn.ReLU(), nn.Dropout(0.1), nn.Linear(d, 1))
        # start near the observed rates: P(Y>=1) ~2%, P(Y>=2) ~0.6%, P(Y>=3) ~0.13%
        self.theta0, self.dtheta = nn.Parameter(torch.tensor(4.0)), nn.Parameter(torch.tensor([0.87, 1.28]))

    def thresholds(self):                               # theta_1 < theta_2 < theta_3 by construction
        return torch.cat([self.theta0.view(1), self.theta0 + torch.cumsum(Fn.softplus(self.dtheta), 0)])

    def forward(self, b):
        parts = [self.feat(b['X'])]
        if self.cfg['emb']:
            c = self.item(E[b['doc']].float())                                    # B, M, h
            q = self.qry(QE[b['quser']].float())[:, None].expand(-1, c.shape[1], -1)
            parts += [c, q]
        if self.cfg['history']:
            hv = self.hist(torch.cat([self.item(E[b['hdoc']].float()), self.type_emb(b['htype']),
                                      self.age(b['hage'][..., None])], -1))     # B, L, h
            pre = (self.att_h(hv)[:, None] + self.att_c(c)[:, :, None]
                   + self.att_p(hv[:, None] * c[:, :, None]))                     # B, M, L, h
            a = self.att_out(torch.relu(pre)).squeeze(-1).float()
            a = a.masked_fill(~b['hmask'][:, None, :], -1e4)
            w = torch.softmax(a, -1) * b['hmask'].any(-1)[:, None, None]          # no history -> zero interest
            parts.append(torch.einsum('bml,bld->bmd', w.to(hv.dtype), hv))
        z = self.enc(torch.cat(parts, -1))
        if self.cfg['slate_ctx']:
            m = b['mask'][..., None].to(z.dtype)
            ctx = ((z * m).sum(1, keepdim=True) / m.sum(1, keepdim=True).clamp(min=1)).expand_as(z)
            z = torch.cat([z, ctx, z - ctx], -1)
        return self.head(z).squeeze(-1)                                           # score s, B x M

def make_batch(S, slates, X):
    st, sz = S['start'][slates], S['size'][slates]
    B, M = len(slates), int(sz.max())
    local = np.concatenate([np.arange(a, a + n) for a, n in zip(st, sz)])
    rows = S['idx'][local]
    rows_t = torch.from_numpy(rows)
    bi = torch.from_numpy(np.repeat(np.arange(B), sz)); pi = torch.from_numpy(local - np.repeat(st, sz))
    def pad(src, fill=0):
        out = torch.full((B, M) + tuple(src.shape[1:]), fill, dtype=src.dtype)
        out[bi, pi] = src[rows_t]
        return out
    users = S['user'][slates]
    mask = torch.zeros(B, M, dtype=torch.bool); mask[bi, pi] = True
    b = {'X': pad(X), 'doc': pad(DOC_T), 'y': pad(LABEL_T), 'mask': mask, 'quser': torch.from_numpy(users),
         'hdoc': H_DOC_T[users], 'htype': H_TYPE_T[users], 'hage': H_AGE_T[users], 'hmask': H_MASK_T[users]}
    b = {k: v.to(DEVICE, non_blocking=True) for k, v in b.items()}
    b['rows'] = rows
    return b

def loss_fn(s, b, theta):
    m, y = b['mask'], b['y']
    tgt = (y[..., None] >= torch.arange(1, 4, device=s.device)).float()
    bce = Fn.binary_cross_entropy_with_logits(s[..., None] - theta, tgt, reduction='none').sum(-1)
    bce = (bce * m).sum() / m.sum()
    gain = (2.0 ** y - 1) * m
    p = gain / gain.sum(1, keepdim=True).clamp(min=1e-9)
    listwise = -(p * torch.log_softmax(s.masked_fill(~m, -1e4), 1)).sum(1).mean()
    return bce + listwise

print('model and batching defined')

model and batching defined


## 6. Training and evaluation

AdamW (lr 1e-3), mixed precision on GPU, validation NDCG@10 after every epoch; the best epoch's weights are kept. Evaluation is identical to every earlier step (per slate, random tie-breaking, gain 2^label − 1), so the numbers are directly comparable with LightGBM.

In [7]:
def evaluate(u, lab, score, k=10, seed=0):
    lab = lab.astype(np.float64)
    r = np.random.default_rng(seed).random(len(u))
    order = np.lexsort((r, -np.asarray(score, dtype=np.float64), u))
    u_s, lab_s = u[order], lab[order]
    start = np.r_[0, np.flatnonzero(u_s[1:] != u_s[:-1]) + 1]
    size = np.diff(np.r_[start, len(u_s)])
    rank = np.arange(len(u_s)) - np.repeat(start, size)
    top = rank < k
    disc = np.where(top, 1.0 / np.log2(rank + 2), 0.0)
    gid = np.repeat(np.arange(len(start)), size)
    dcg = np.bincount(gid, weights=(2 ** lab_s - 1) * disc)
    idcg = np.bincount(gid, weights=(2 ** lab[np.lexsort((-lab, u))] - 1) * disc)
    has_pos = np.bincount(gid, weights=(lab_s > 0)) > 0
    has_buy = np.bincount(gid, weights=(lab_s == 3)) > 0
    hit = np.bincount(gid, weights=((lab_s > 0) & top)) > 0
    hit_buy = np.bincount(gid, weights=((lab_s == 3) & top)) > 0
    return {'NDCG@10': (dcg[has_pos] / idcg[has_pos]).mean(), 'HR@10': hit[has_pos].mean(),
            'Purchase-HR@10': hit_buy[has_buy].mean(), 'slates': int(has_pos.sum())}

@torch.no_grad()
def predict(model, S, X, bs=128):
    model.eval()
    out = np.empty(len(S['idx']), np.float32)
    pos_of = np.empty(len(USER), np.int64); pos_of[S['idx']] = np.arange(len(S['idx']))
    for i in range(0, len(S['start']), bs):
        b = make_batch(S, np.arange(i, min(i + bs, len(S['start']))), X)
        with torch.autocast(device_type='cuda' if USE_AMP else 'cpu', dtype=torch.float16, enabled=USE_AMP):
            s = model(b)
        out[pos_of[b['rows']]] = s.float()[b['mask']].cpu().numpy()
    return out

def eval_split(model, name, X):
    S = SPLITS[name]
    score = predict(model, S, X)
    return evaluate(USER[S['idx']], LABEL[S['idx']], score), score

def train(cfg):
    X = get_X(cfg['features'])
    torch.manual_seed(SEED)
    model = Ranker(X.shape[1], cfg).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
    scaler = torch.amp.GradScaler(enabled=USE_AMP)
    rng = np.random.default_rng(SEED)
    TR = SPLITS['train']; best, best_state, log = -1.0, None, []
    for ep in range(EPOCHS):
        model.train(); t0 = time.time(); tot = 0.0
        perm = rng.permutation(len(TR['start']))
        for i in range(0, len(perm), BATCH):
            b = make_batch(TR, perm[i:i + BATCH], X)
            with torch.autocast(device_type='cuda' if USE_AMP else 'cpu', dtype=torch.float16, enabled=USE_AMP):
                s = model(b)
            loss = loss_fn(s.float(), b, model.thresholds())
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            tot += loss.item()
        val, _ = eval_split(model, 'val', X)
        log.append({'epoch': ep + 1, 'loss': tot / (len(perm) / BATCH), 'val NDCG@10': val['NDCG@10'], 'min': (time.time() - t0) / 60})
        print(f"  epoch {ep + 1}: loss {log[-1]['loss']:.4f} | val NDCG@10 {val['NDCG@10']:.4f} | {log[-1]['min']:.1f} min")
        if val['NDCG@10'] > best:
            best, best_state = val['NDCG@10'], copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    return model, pd.DataFrame(log)

In [8]:
RUNS = {   # each run adds one component; comment out runs to save GPU time
    'A. MLP on 32 features':               {'features': 'lean', 'emb': False, 'history': False, 'slate_ctx': False},
    'B. + embeddings & history attention':  {'features': 'lean', 'emb': True,  'history': True,  'slate_ctx': False},
    'C. + slate context (full model)':      {'features': 'lean', 'emb': True,  'history': True,  'slate_ctx': True},
    'D. full model, all 55 features':       {'features': 'all',  'emb': True,  'history': True,  'slate_ctx': True},
}
results, test_scores, models = {}, {}, {}
for name, cfg in RUNS.items():
    print(f'== {name}')
    t0 = time.time()
    model, log = train(cfg)
    res, score = eval_split(model, 'test', get_X(cfg['features']))
    res['train min'] = (time.time() - t0) / 60
    results[name], test_scores[name], models[name] = res, score, model
    print(f"  test NDCG@10 {res['NDCG@10']:.4f} | HR@10 {res['HR@10']:.4f} | Purchase-HR@10 {res['Purchase-HR@10']:.4f}")
    if USE_AMP:
        torch.cuda.empty_cache()

table = pd.DataFrame(results).T
table.loc['LightGBM benchmark (Step 4b)'] = pd.Series(LGBM_REF)
table[['NDCG@10', 'HR@10', 'Purchase-HR@10', 'slates', 'train min']].round(4)

== A. MLP on 32 features
  epoch 1: loss 3.5802 | val NDCG@10 0.4078 | 0.3 min
  epoch 2: loss 3.5430 | val NDCG@10 0.4101 | 0.3 min
  epoch 3: loss 3.5291 | val NDCG@10 0.4087 | 0.3 min
  epoch 4: loss 3.5192 | val NDCG@10 0.4098 | 0.3 min
  test NDCG@10 0.4110 | HR@10 0.7150 | Purchase-HR@10 0.6753
== B. + embeddings & history attention
  epoch 1: loss 3.5700 | val NDCG@10 0.4076 | 0.6 min
  epoch 2: loss 3.5305 | val NDCG@10 0.4106 | 0.6 min
  epoch 3: loss 3.5156 | val NDCG@10 0.4108 | 0.6 min
  epoch 4: loss 3.5014 | val NDCG@10 0.4115 | 0.6 min
  test NDCG@10 0.4132 | HR@10 0.7178 | Purchase-HR@10 0.6724
== C. + slate context (full model)
  epoch 1: loss 3.5596 | val NDCG@10 0.4095 | 0.6 min
  epoch 2: loss 3.5177 | val NDCG@10 0.4121 | 0.6 min
  epoch 3: loss 3.4978 | val NDCG@10 0.4118 | 0.6 min
  epoch 4: loss 3.4811 | val NDCG@10 0.4136 | 0.6 min
  test NDCG@10 0.4150 | HR@10 0.7197 | Purchase-HR@10 0.6756
== D. full model, all 55 features
  epoch 1: loss 3.5595 | val NDCG@10

,NDCG@10,HR@10,Purchase-HR@10,slates,train min
A. MLP on 32 features,0.4110,0.7150,0.6753,25822.0,1.5803
B. + embeddings & history attention,0.4132,0.7178,0.6724,25822.0,2.4911
C. + slate context (full model),0.4150,0.7197,0.6756,25822.0,2.5925
"D. full model, all 55 features",0.4154,0.7187,0.6724,25822.0,3.0589
LightGBM benchmark (Step 4b),0.4159,0.7196,0.6692,NaN,NaN


## 7. Where does each model win? Cold-candidate segment and ordinal output

Same segmentation as Steps 3–4b: slates grouped by whether their positives are all cold items (in no user's history), mixed, or all warm.

In [9]:
TE = SPLITS['test']
te_u, te_lab, te_wid = USER[TE['idx']], LABEL[TE['idx']], feat['wid'].to_numpy()[TE['idx']]
hist_wids = np.unique(pd.read_parquet(OUT_DIR / 'events.parquet', columns=['wid'])['wid'].to_numpy())
j = np.searchsorted(hist_wids, te_wid)
cold = (j >= len(hist_wids)) | (hist_wids[np.minimum(j, len(hist_wids) - 1)] != te_wid)
cold_pos = pd.Series(np.where(te_lab > 0, cold, np.nan))
seg_user = pd.cut(cold_pos.groupby(te_u).mean(), [-0.01, 0.0, 0.999, 1.0], labels=['all warm', 'mixed', 'all cold'])
seg = pd.Series(te_u).map(seg_user).to_numpy()

rows = {}
for name, score in test_scores.items():
    rows[name] = {s: evaluate(te_u[seg == s], te_lab[seg == s], score[seg == s])['NDCG@10'] for s in ['all warm', 'mixed', 'all cold']}
seg_table = pd.DataFrame(rows).T
seg_table.loc['LightGBM benchmark (Step 4b)'] = {'all warm': 0.5080, 'mixed': 0.2685, 'all cold': 0.1138}
print('test NDCG@10 by segment:')
print(seg_table.round(4).to_string())

best_name = max(results, key=lambda n: results[n]['NDCG@10'])
theta = models[best_name].thresholds().detach().float().cpu().numpy()
s = test_scores[best_name]
P = 1 / (1 + np.exp(-(s[:, None] - theta)))                      # P(Y>=1), P(Y>=2), P(Y>=3)
calib = pd.DataFrame({'predicted mean': P.mean(0), 'observed rate': [(te_lab >= k).mean() for k in (1, 2, 3)]},
                     index=['P(Y>=1) click or more', 'P(Y>=2) cart or more', 'P(Y>=3) purchase'])
print(f'\nordinal output of the best model ({best_name}), thresholds {np.round(theta, 2)}:')
calib.round(4)

test NDCG@10 by segment:
                                     all warm   mixed  all cold
A. MLP on 32 features                  0.5024  0.2656    0.1097
B. + embeddings & history attention    0.5051  0.2675    0.1103
C. + slate context (full model)        0.5065  0.2687    0.1147
D. full model, all 55 features         0.5067  0.2671    0.1172
LightGBM benchmark (Step 4b)           0.5080  0.2685    0.1138

ordinal output of the best model (D. full model, all 55 features), thresholds [3.51 4.79 6.38]:


,predicted mean,observed rate
P(Y>=1) click or more,0.0212,0.0205
P(Y>=2) cart or more,0.0064,0.0062
P(Y>=3) purchase,0.0014,0.0014


In [10]:
out = table.copy(); out.index.name = 'model'
safe_out = OUT_DIR / 'step5_results.csv'
out.to_csv(safe_out.with_suffix('.tmp')); os.replace(safe_out.with_suffix('.tmp'), safe_out)
print('saved', safe_out)
try:
    drive.flush_and_unmount()
    print('Drive flushed and unmounted - safe to close the runtime')
except NameError:
    pass

saved /content/drive/MyDrive/JDSearch/Raw Data/processed/step5_results.csv
Drive flushed and unmounted - safe to close the runtime


## 8. Takeaways

*(filled in after the run)*